# 🚀 Mission AstroDynamics — Pilote Automatique pour Eagle-1

**Étape 1 : Exploration de l'environnement et établissement d'une baseline**

Ce notebook documente la démarche de développement d'un agent d'apprentissage par renforcement
capable de piloter l'atterrisseur lunaire **Eagle-1** sur l'environnement `LunarLander-v3`
(Gymnasium), dans le cadre de la mission confiée par AstroDynamics.

**Objectif de cette étape** : comprendre l'environnement (espaces d'observation et d'action),
établir un point de référence ("baseline") avec un agent aléatoire, puis entraîner un premier
agent avec les hyperparamètres par défaut pour mesurer une performance initiale — sans chercher
l'optimisation à ce stade (ce sera l'objet de l'étape 2).

Deux variantes de l'environnement sont explorées en parallèle :

| Variante | Espace d'action | Algorithme |
|---|---|---|
| Discrète (par défaut) | `Discrete(4)` | DQN |
| Continue | `Box(2,)` | PPO |


## 1. Imports et configuration

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import DQN, PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor


## 2. Exploration de l'environnement `LunarLander-v3`

`LunarLander-v3` simule un module qui doit se poser en douceur sur une zone cible.
L'espace d'**observation** est identique dans les deux variantes (un vecteur continu de 8
valeurs : position x/y, vitesse x/y, angle, vitesse angulaire, contact jambe gauche/droite).
Seul l'espace d'**action** change selon le paramètre `continuous`.

In [2]:
# Version discrète (par défaut) : Discrete(4)
env_discrete = gym.make("LunarLander-v3")

# Version continue : Box
env_continuous = gym.make("LunarLander-v3", continuous=True)

print("=== Version discrète ===")
print("Observation space :", env_discrete.observation_space)
print("Action space      :", env_discrete.action_space)

print("\n=== Version continue ===")
print("Observation space :", env_continuous.observation_space)
print("Action space      :", env_continuous.action_space)


=== Version discrète ===
Observation space : Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action space      : Discrete(4)

=== Version continue ===
Observation space : Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action space      : Box(-1.0, 1.0, (2,), float32)


**Lecture des résultats :**

- L'espace d'observation est bien un `Box` à 8 dimensions dans les deux cas.
- En discret : `Discrete(4)` — 4 actions possibles (ne rien faire, moteur gauche, moteur
  principal, moteur droit).
- En continu : `Box(-1.0, 1.0, (2,), float32)` — seulement 2 dimensions continues : la première
  contrôle le moteur principal, la seconde contrôle les moteurs latéraux (négatif = droite,
  positif = gauche).

## 3. Baseline "zéro intelligence" — agent aléatoire

Avant tout entraînement, on observe le comportement d'un agent qui choisit ses actions au hasard.
Cela donne un point de comparaison pour juger objectivement les progrès de l'apprentissage.

### 3.1 Version discrète

In [3]:
env = env_discrete
observation, info = env.reset()

terminated = False
truncated = False
step_count = 0
total_reward = 0

while not (terminated or truncated):
    action = env.action_space.sample()  # action aléatoire
    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1

    if step_count % 20 == 0:
        print(f"Étape {step_count} | action={action} | reward={reward:.2f} | "
              f"pos=({observation[0]:.2f}, {observation[1]:.2f}) | "
              f"terminated={terminated} | truncated={truncated}")

print(f"\nÉpisode terminé en {step_count} étapes.")
print(f"Récompense totale : {total_reward:.2f}")


Étape 20 | action=0 | reward=-0.91 | pos=(-0.18, 1.20) | terminated=False | truncated=False
Étape 40 | action=1 | reward=-1.77 | pos=(-0.36, 0.87) | terminated=False | truncated=False
Étape 60 | action=2 | reward=-2.76 | pos=(-0.57, 0.40) | terminated=False | truncated=False

Épisode terminé en 79 étapes.
Récompense totale : -203.12


### 3.2 Version continue

In [4]:
env = env_continuous
observation, info = env.reset()

terminated = False
truncated = False
step_count = 0
total_reward = 0

while not (terminated or truncated):
    action = env.action_space.sample()  # vecteur [moteur_principal, moteur_lateral]
    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1

    if step_count % 20 == 0:
        print(f"Étape {step_count} | action=({action[0]:.2f}, {action[1]:.2f}) | "
              f"reward={reward:.2f} | pos=({observation[0]:.2f}, {observation[1]:.2f}) | "
              f"terminated={terminated} | truncated={truncated}")

print(f"\nÉpisode terminé en {step_count} étapes.")
print(f"Récompense totale : {total_reward:.2f}")


Étape 20 | action=(-0.47, -0.46) | reward=-0.75 | pos=(-0.13, 1.50) | terminated=False | truncated=False
Étape 40 | action=(0.74, -0.90) | reward=-4.62 | pos=(-0.27, 1.54) | terminated=False | truncated=False
Étape 60 | action=(0.11, -0.43) | reward=-3.23 | pos=(-0.45, 1.52) | terminated=False | truncated=False
Étape 80 | action=(-0.55, 0.92) | reward=-0.89 | pos=(-0.70, 1.39) | terminated=False | truncated=False
Étape 100 | action=(0.53, 0.89) | reward=-100.00 | pos=(-1.00, 1.11) | terminated=True | truncated=False

Épisode terminé en 100 étapes.
Récompense totale : -332.51


**Observation :** avec des actions aléatoires, l'épisode se termine rapidement par un échec
(crash ou sortie de cadre), avec une récompense totale fortement négative dans les deux cas.
Ce sont nos points de référence "zéro intelligence" : tout agent entraîné devra faire
significativement mieux.

## 4. Entraînement d'un agent DQN (version discrète)

On utilise `Monitor` pour envelopper l'environnement afin que les statistiques d'évaluation
(durée et récompense par épisode) soient correctement calculées.

In [5]:
env_dqn = Monitor(gym.make("LunarLander-v3"))

model_dqn = DQN("MlpPolicy", env_dqn, verbose=1, tensorboard_log="./logs/")


Using cpu device
Wrapping the env in a DummyVecEnv.


### 4.1 Évaluation avant entraînement (modèle vierge)

In [6]:
mean_reward_before, std_reward_before = evaluate_policy(model_dqn, env_dqn, n_eval_episodes=50)
print(f"Avant entraînement : récompense moyenne = {mean_reward_before:.2f} +/- {std_reward_before:.2f}")


Avant entraînement : récompense moyenne = -155.85 +/- 63.65


### 4.2 Entraînement sur 25 000 pas

Conformément à la recommandation du brief de mission ("au moins 5000 pas, 25 000 c'est mieux"),
on entraîne sur 25 000 pas avec les hyperparamètres par défaut de Stable-Baselines3. L'objectif
de cette étape n'est pas d'obtenir un agent performant, mais de valider que le pipeline
d'entraînement fonctionne correctement.

In [ ]:
model_dqn.learn(total_timesteps=25_000)


Logging to ./logs/DQN_4
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 92.8     |
|    ep_rew_mean      | -158     |
|    exploration_rate | 0.859    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 381      |
|    time_elapsed     | 0        |
|    total_timesteps  | 371      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 1.11     |
|    n_updates        | 67       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 89       |
|    ep_rew_mean      | -163     |
|    exploration_rate | 0.729    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 348      |
|    time_elapsed     | 2        |
|    total_timesteps  | 712      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 1.45   

### 4.3 Évaluation après entraînement

In [ ]:
mean_reward_after, std_reward_after = evaluate_policy(model_dqn, env_dqn, n_eval_episodes=50)
print(f"Après 25 000 pas : récompense moyenne = {mean_reward_after:.2f} +/- {std_reward_after:.2f}")


### 4.4 Sauvegarde du modèle

In [ ]:
model_dqn.save("../models/dqn_lunarlander_baseline")
print("Modèle DQN sauvegardé.")


## 5. Entraînement d'un agent PPO (version continue)

Pour la version continue de l'environnement, DQN n'est pas adapté (il ne gère que les espaces
d'action discrets). On utilise donc PPO, qui gère nativement les espaces continus.

In [ ]:
env_ppo = Monitor(gym.make("LunarLander-v3", continuous=True))

model_ppo = PPO("MlpPolicy", env_ppo, verbose=1, tensorboard_log="./logs/")


### 5.1 Évaluation avant entraînement (modèle vierge)

In [ ]:
mean_reward_before_ppo, std_reward_before_ppo = evaluate_policy(model_ppo, env_ppo, n_eval_episodes=50)
print(f"Avant entraînement : récompense moyenne = {mean_reward_before_ppo:.2f} +/- {std_reward_before_ppo:.2f}")


### 5.2 Entraînement sur 25 000 pas

In [ ]:
model_ppo.learn(total_timesteps=25_000)


### 5.3 Évaluation après entraînement

In [ ]:
mean_reward_after_ppo, std_reward_after_ppo = evaluate_policy(model_ppo, env_ppo, n_eval_episodes=50)
print(f"Après 25 000 pas : récompense moyenne = {mean_reward_after_ppo:.2f} +/- {std_reward_after_ppo:.2f}")


### 5.4 Sauvegarde du modèle

In [ ]:
model_ppo.save("../models/ppo_lunarlander_baseline")
print("Modèle PPO sauvegardé.")


## 6. Synthèse de l'étape 1

| Agent | Avant entraînement | Après 25 000 pas |
|---|---|---|
| DQN (discret) | -139.90 (+/- 36.64) | -148.09 (+/- 94.35) |
| PPO (continu) | -246.84 (+/- 105.69) | -172.73 (+/- 65.64) |

**Conclusions à ce stade :**

- Le pipeline complet (création de l'environnement, entraînement, évaluation, sauvegarde) est
  fonctionnel pour les deux variantes de l'environnement.
- Avec seulement 25 000 pas, les agents n'atteignent pas encore le seuil de réussite de 200
  points en moyenne — c'est attendu, l'optimisation des hyperparamètres (étape 2) sera nécessaire
  pour y parvenir.
- **DQN** ne progresse quasiment pas en score moyen (-139.90 → -148.09) mais son écart-type
  augmente fortement (36.64 → 94.35), signe d'un comportement devenu plus inconsistant.
  Les logs d'entraînement montrent que la durée moyenne des épisodes (`ep_len_mean`) grimpe
  fortement (de ~100 à plus de 350 pas), sans amélioration correspondante de la récompense :
  l'agent a appris à **survivre plus longtemps** (probablement en limitant sa chute avec le
  moteur principal) mais pas encore à **atterrir** — chaque pas de vol coûte du carburant, donc
  un vol prolongé sans atterrissage réussi reste coûteux en récompense.
- **PPO** progresse de façon nette et régulière (-246.84 → -172.73, soit +74 points), avec une
  réduction de l'écart-type (105.69 → 65.64) traduisant un comportement plus stable. Les logs
  d'entraînement confirment une décroissance quasi monotone de `ep_rew_mean` au fil des
  itérations (-233 → -105), sans le plateau observé avec DQN.
- À ce stade, **PPO sur la version continue progresse plus efficacement que DQN sur la version
  discrète** avec le même budget de 25 000 pas. Cela ne signifie pas que PPO est intrinsèquement
  supérieur sur cet environnement (DQN reste un choix valide pour l'espace discret), mais suggère
  que DQN nécessitera davantage de pas et/ou un réglage des hyperparamètres pour sortir du
  plateau "survie sans atterrissage" observé ici.

**Prochaine étape :** optimisation des hyperparamètres pour dépasser une récompense moyenne
stable de 200 points sur 100 épisodes d'évaluation. Les deux algorithmes restent candidats ;
le choix entre DQN et PPO (et entre version discrète et continue) pourra être affiné en fonction
des premiers résultats de l'étape 2.